# GxP-LLM stage evaluation

Run this notebook once per baseline, fine-tuned, or quantized model. Attach the versioned `gxp-source` and `gxp-data` datasets, enable Internet, and add the relevant Kaggle Secrets.

In [ ]:
%pip install -q torch peft bitsandbytes accelerate torchvision sentence-transformers rouge-score nltk litellm wandb
%pip install --upgrade transformers

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print(torch.cuda.get_device_name(0))

In [ ]:
# Attach `gxp-source` (this repository) and `gxp-data` in Kaggle's Input pane.
import json, os, sys
from pathlib import Path

root = Path("/kaggle/input/datasets/yuvalmehta")
print([p.name for p in root.iterdir()])

SOURCE_DIR = root / 'gxp-source'
DATA_DIR = root / 'gxp-data'
assert (SOURCE_DIR / 'eval' / 'run.py').exists(), 'Attach the gxp-source dataset.'
assert (DATA_DIR / 'eval.jsonl').exists(), 'Attach the gxp-data dataset.'
sys.path.insert(0, str(SOURCE_DIR))

# Change only these values for each run. For GPTQ/AWQ set LOAD_IN_4BIT=False.
STAGE = 'baseline'  # baseline | finetuned | gptq_4bit | awq_4bit
MODEL_PATH = 'unsloth/Qwen3.5-4B'
ADAPTER_PATH = None
LOAD_IN_4BIT = True
JUDGE_MODEL = 'gemini/gemini-3.6-flash-lite'  # or groq/openai/gpt-oss-20b, nvidia_nim/<model>, or None
EVAL_BATCH_SIZE = 4  # Lower to 2 if a T4 runs out of memory.
MAX_NEW_TOKENS = 512  # 512 is more thorough but roughly twice as slow.
JUDGE_CONCURRENCY = 1  # Gemini free-tier requests are rate-limited; keep this at 1.
JUDGE_MAX_RETRIES = 4
JUDGE_INITIAL_BACKOFF_SECONDS = 5
OUTPUT_DIR = Path('/kaggle/working') / f'gxp_eval_{STAGE}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = {'stage': STAGE, 'model_path': MODEL_PATH, 'adapter_path': ADAPTER_PATH, 'load_in_4bit': LOAD_IN_4BIT, 'judge_model': JUDGE_MODEL, 'batch_size': EVAL_BATCH_SIZE, 'max_new_tokens': MAX_NEW_TOKENS, 'judge_concurrency': JUDGE_CONCURRENCY, 'judge_max_retries': JUDGE_MAX_RETRIES, 'judge_initial_backoff_seconds': JUDGE_INITIAL_BACKOFF_SECONDS, 'data_dir': str(DATA_DIR)}
(OUTPUT_DIR / 'experiment_config.json').write_text(json.dumps(config, indent=2))

In [ ]:
# Kaggle secrets: WANDB_API_KEY plus the key for your selected judge.
from kaggle_secrets import UserSecretsClient
import wandb

def load_secret(name, required=False):
    try:
        value = UserSecretsClient().get_secret(name)
        if value:
            os.environ[name] = value
            return value
    except Exception:
        pass
    if required:
        raise RuntimeError(f'Add {name} as a Kaggle Secret.')

load_secret('WANDB_API_KEY', required=True)
provider_secret = {'gemini/': 'GEMINI_API_KEY', 'groq/': 'GROQ_API_KEY', 'nvidia_nim/': 'NVIDIA_API_KEY'}
if JUDGE_MODEL:
    secret_name = next((v for k, v in provider_secret.items() if JUDGE_MODEL.startswith(k)), None)
    if not secret_name:
        raise ValueError('Use a supported provider-qualified judge model or set JUDGE_MODEL=None.')
    load_secret(secret_name, required=True)

run = wandb.init(project='gxp-llm', name=f'{STAGE}-{MODEL_PATH.rsplit("/", 1)[-1]}', group=MODEL_PATH.rsplit('/', 1)[-1], job_type='eval', tags=[STAGE], config=config)

In [ ]:
from eval.run import run_full_eval

results = run_full_eval(model_path=MODEL_PATH, adapter_path=ADAPTER_PATH, data_dir=str(DATA_DIR), judge_model=JUDGE_MODEL, output_dir=str(OUTPUT_DIR), load_in_4bit=LOAD_IN_4BIT, batch_size=EVAL_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS, judge_concurrency=JUDGE_CONCURRENCY, judge_max_retries=JUDGE_MAX_RETRIES, judge_initial_backoff_seconds=JUDGE_INITIAL_BACKOFF_SECONDS)

for split_name, result in results.items():
    metrics = result.metrics
    wandb.log({f'{split_name}/exact_match': metrics.get('exact_match', 0), f'{split_name}/rougeL_f1': metrics.get('rougeL', {}).get('rougeL_f1', 0), f'{split_name}/bleu': metrics.get('bleu', 0)})
    for category, scores in result.judge_scores.items():
        wandb.log({f'{split_name}/judge/{category}/{metric}': value for metric, value in scores.items()})
    if result.adversarial:
        wandb.log({f'{split_name}/{metric}': value for metric, value in result.adversarial.items() if isinstance(value, (int, float))})

In [ ]:
artifact = wandb.Artifact(f'eval-{STAGE}-{MODEL_PATH.rsplit("/", 1)[-1]}', type='evaluation', metadata=config)
artifact.add_dir(str(OUTPUT_DIR))
wandb.log_artifact(artifact)
wandb.finish()
print(f'Published Kaggle output: {OUTPUT_DIR}')